# Merge and validate two deterministic final-test shards

Run after both A100 shard notebooks report completion. This notebook needs only a CPU runtime. It restores original 2,000-row order, rejects missing/duplicate/non-integer answers, and records SHA-256 values.

In [ ]:
# Cell 1 — Mount Drive and locate the original input and both isolated runs.
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
FULL_INPUT=Path("/content/drive/MyDrive/test_submission.csv")
RUN_ROOT=Path("/content/drive/MyDrive/2026소중한챌린지/runs/FINAL-0006-r3-adaptive-pal3-deterministic-2shard")
SHARD_DIRS=[RUN_ROOT/"shard_0_of_2",RUN_ROOT/"shard_1_of_2"]
ALLOWED_POLICY_COMMITS={
    "24380f2ab9f4a4d7af417193b44b2fa0da22b7ed",  # deterministic adaptive code; runtime config overridden to chunk 4
    "5190f774e8e988d5471d910f7c581fc205397d53",  # same code with chunk 4 as repository default
}
REQUIRED_8192_CHUNK=4
assert FULL_INPUT.exists() and all(path.exists() for path in SHARD_DIRS)
print("[RUN ROOT]",RUN_ROOT)

In [ ]:
# Cell 2 — Strictly validate shard boundaries, merge by original ID order, and save.
import hashlib,json,re,pandas as pd
def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()
full=pd.read_csv(FULL_INPUT,dtype=str,keep_default_na=False);full.columns=[str(c).lstrip("\ufeff").strip() for c in full.columns]
assert len(full)==2000 and full.id.is_unique and {"id","question"}.issubset(full.columns)
parts=[];input_parts=[]
for index,path in enumerate(SHARD_DIRS):
    shard_input=pd.read_csv(path/f"input_shard_{index}_of_2.csv",dtype=str,keep_default_na=False)
    part=pd.read_csv(path/"submissions/submission.csv",dtype=str,keep_default_na=False)
    assert list(part.columns)==["id","answer"] and len(part)==1000 and part.id.is_unique
    assert part.id.tolist()==shard_input.id.tolist()==full.iloc[index*1000:(index+1)*1000].id.tolist()
    assert part.answer.str.fullmatch(r"-?\d+").all()
    input_parts.append(shard_input);parts.append(part)
stacked=pd.concat(parts,ignore_index=True);assert len(stacked)==2000 and stacked.id.is_unique
assert stacked.id.tolist()==full.id.tolist() and set(stacked.id)==set(full.id)
submission=stacked[["id","answer"]].copy();assert submission.answer.str.fullmatch(r"-?\d+").all()
SUBMISSION_DIR=RUN_ROOT/"submissions";REPORT_DIR=RUN_ROOT/"reports"
SUBMISSION_DIR.mkdir(parents=True,exist_ok=True);REPORT_DIR.mkdir(parents=True,exist_ok=True)
SUBMISSION_PATH=SUBMISSION_DIR/"submission.csv";submission.to_csv(SUBMISSION_PATH,index=False,encoding="utf-8")
roundtrip=pd.read_csv(SUBMISSION_PATH,dtype=str,keep_default_na=False);assert roundtrip.equals(submission)
EASY_COPY=Path("/content/drive/MyDrive/submission_final_r3_2shard.csv");submission.to_csv(EASY_COPY,index=False,encoding="utf-8")
print("[MERGED]",len(submission));print("[SUBMISSION]",SUBMISSION_PATH);print("[EASY COPY]",EASY_COPY)

In [ ]:
# Cell 3 — Write the final merge and reproducibility report.
reports=[json.loads((path/"reports/final_inference_report.json").read_text()) for path in SHARD_DIRS]
for index,report in enumerate(reports):
    assert report["status"]=="completed" and report["rows"]==1000
    assert report["base_model"]=="Qwen/Qwen2.5-3B-Instruct"
    assert report["reproducibility"]["git"]["commit"] in ALLOWED_POLICY_COMMITS
    assert report["configuration"]["adaptive_length"]["prompt_chunk_8192"]==REQUIRED_8192_CHUNK
    assert report["reproducibility"]["vllm_batch_invariant"]=="1"
    assert report["reproducibility"]["async_scheduling"] is False
observed_commits=sorted({item["reproducibility"]["git"]["commit"] for item in reports})
report={
    "status":"completed","rows":2000,"policy_commits":observed_commits,"prompt_chunk_8192":REQUIRED_8192_CHUNK,
    "input":str(FULL_INPUT),"input_sha256":sha256_file(FULL_INPUT),
    "shards":[{"index":i,"submission":str(path/"submissions/submission.csv"),"submission_sha256":sha256_file(path/"submissions/submission.csv"),"input_sha256":sha256_file(path/f"input_shard_{i}_of_2.csv")} for i,path in enumerate(SHARD_DIRS)],
    "submission":str(SUBMISSION_PATH),"submission_sha256":sha256_file(SUBMISSION_PATH),
    "id_order_matches_original":True,"duplicate_ids":0,"non_integer_answers":0,
    "external_api_calls":0,"answer_lookup":False,"test_labels_read":False
}
REPORT_PATH=REPORT_DIR/"two_shard_merge_report.json";REPORT_PATH.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(report,ensure_ascii=False,indent=2));print("[REPORT]",REPORT_PATH)